# Step1 — Ingesta y armonización de la base histórica de eventos (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 1 — Construcción de la base histórica (`Hazard_Event`)
**Autor:** Jerson Achicanoy (FAO OER) · implementación asistida
**Fecha:** 2026-08-05

**Objetivo de este notebook:** leer la base cruda de eventos de desastre de
SNGRE (`BD_2015_2025`, 69,718 registros, hoja del workbook maestro
`Indices ENSO (1).xlsx`), filtrarla al alcance de peligro acordado
(inundación fluvial/pluvial, aluvión, deslizamiento, socavamiento),
armonizarla, enriquecerla con el índice ENSO mensual (ICEN/ONI) y exportarla
como la tabla `Hazard_Event` limpia que alimentará las fases 2–8.

**No calcula** índices ni umbrales de cronicidad — eso es Fase 3–8. Este
notebook solo prepara datos limpios y trazables.

Cada celda de código está comentada línea por línea, conforme a la regla del
proyecto del 2026-08-05. La lógica vive aquí (no en `src/`) por decisión
explícita del mismo día; `src/utilidades` solo aporta helpers transversales
(config, logging, E/S).

Ver justificación metodológica completa en `docs/methodology/01_data_ingestion.md`
y el resumen para no-técnicos en `docs/executive/01_data_ingestion.md`.

**English:** Hazard Event Ingestion

Builds the historical hazard-event table (2015-2025) from the raw SNGRE/ECU911 workbook: filters the hazards in scope, validates dates and coordinates, removes personal data, builds the complete monthly ENSO series and joins both.

Outputs: `hazard_event` and `enso_index_monthly` in `data/processed/`.

## 0. Configuración del entorno

**English:** 0. Setup

In [ ]:
# Este notebook NO requiere Google Earth Engine ni Copernicus CDS: la Fase 1
# trabaja exclusivamente sobre el workbook tabular ya existente. Las reglas
# de autenticación GEE/CDS del proyecto aplican a partir de la Fase 2+
# (capas ráster), no aquí.

# Standard library: system path handling
import sys  # necesario para añadir la raíz del proyecto a sys.path
# Object-oriented file paths
from pathlib import Path  # manejo de rutas multiplataforma

# Notebook vive en <repo>/notebooks/, así que la raíz del repo es el padre.
# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent  # asume ejecución con cwd = notebooks/
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))  # permite `from src.utilidades import ...`

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path  # carga de config.yaml
# Project logger factory
from src.utilidades.logging_utils import get_logger  # logger estándar del proyecto
# Helpers to read an Excel sheet and write processed tables
from src.utilidades.io_utils import read_excel_sheet, write_processed_table  # helpers de E/S

# Tabular data handling
import pandas as pd  # manipulación tabular
# Numerical operations
import numpy as np  # operaciones vectorizadas (np.where, comparaciones)
import re  # expresiones regulares para el heurístico de PII en 'Fuente'

# Show up to 30 columns when printing DataFrames
pd.set_option("display.max_columns", 30)  # facilita inspección visual en el notebook

In [ ]:
# Cargar la configuración del proyecto (rutas, alcance de peligro, umbrales,
# nunca hardcodeados en este notebook).
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"  # ruta absoluta al YAML
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)  # dict con toda la configuración

# Logger con salida a consola + archivo en <repo>/logs/hci_pipeline.log
# Create a logger that writes to the logs folder
logger = get_logger("step1_ingestion", config, log_dir=PROJECT_ROOT / "logs")
logger.info("Fase 1 iniciada — configuración cargada desde %s", CONFIG_PATH)

## 1. Carga de la base cruda de eventos (`BD_2015_2025`)

Fuente: hoja `BD_2015_2025` del workbook `Indices ENSO (1).xlsx` (ya existente
en la raíz del proyecto — ver excepción de estructura de carpetas en el
documento técnico). 69,718 eventos, 63 columnas, todos los peligros
registrados por SNGRE/ECU911 entre 2015-01-01 y 2025-12-31.

**English:** 1. Load the raw event table

In [ ]:
# Resolver la ruta absoluta al workbook maestro a partir de config.yaml.
# Path to the raw source workbook
workbook_path = resolve_path(config, "raw_workbook", base_dir=Path.cwd())  # cwd = notebooks/

# Leer la hoja completa de eventos crudos (todas las amenazas, sin filtrar).
# Read the raw hazard-events sheet
raw_events = read_excel_sheet(workbook_path, config["sheets"]["hazard_events"])

# Registrar la forma de la tabla cruda para trazabilidad en el log.
# Log the number of rows and columns loaded
logger.info("Base cruda cargada: %d filas x %d columnas", *raw_events.shape)

# Vista rápida de las primeras filas para inspección visual manual.
# Preview the first three rows
raw_events.head(3)

## 1.1 Corrección de datos puntuales (auditable, no silenciosa)

Se aplican, si existen, las correcciones documentadas en `config.yaml` →
`known_data_corrections` — cada una requiere evidencia y decisión explícita
de Jerson, nunca se "arregla" un valor por sospecha sin verificar. El
workbook original (`Indices ENSO (1).xlsx`) **no se modifica**; la
corrección vive solo en este pipeline, sobre una copia en memoria.

**Corrección aplicada 2026-08-06:** evento ID 51387 (Baños de Agua Santa,
aluvión en Río Negro, 2021-09-05), columna `ANIMALES MUERTOS`: 500,000 →
0. Motivo completo en `config.yaml` y en
`docs/methodology/01_data_ingestion.md` §7 — en resumen: inconsistente con
`ANIMALES AFECTADOS`=0 en la misma fila, no mencionado en la descripción
extensa del evento en SNGRE, y no confirmado por prensa independiente que
sí documenta el evento (3 fallecidos, ~50 personas incomunicadas, pérdidas
~600,000 USD) sin mencionar pérdida de ganado. Este valor era el outlier
que dominaba el Índice de Severidad del pilar Ganadería (Fase 4 §6) y
resultó ser el parámetro de mayor sensibilidad de todo el HCI (Fase 8
§5.3) — se corrige en el origen para que las 8 fases posteriores hereden
el dato correcto automáticamente, en vez de parchear cada tabla derivada
por separado.

**English:** 1.1 Auditable data corrections

In [ ]:
# known_data_corrections: lista de correcciones documentadas — ver config.yaml
# para la evidencia completa de cada una. Se aplican sobre raw_events (nombres
# de columna originales del Excel), antes de cualquier filtro o renombrado,
# para que TODAS las fases posteriores hereden el valor corregido.
# Known data corrections declared in the config
corrections = config.get("known_data_corrections", [])
# Apply each correction one by one
for corr in corrections:
    # Select the event to correct by its ID
    mask = raw_events["ID"] == corr["event_id"]
    # Count matching rows
    n_afectados = int(mask.sum())
    # Each correction must match exactly one event
    assert n_afectados == 1, (
        f"se esperaba exactamente 1 fila con ID={corr['event_id']}, "
        f"se encontraron {n_afectados} — revisar antes de aplicar la corrección"
    )
    # Read the current value
    valor_anterior = raw_events.loc[mask, corr["column"]].iloc[0]
    # Check the current value is the one the correction expects
    assert valor_anterior == corr["original_value"], (
        f"el valor original en el workbook ({valor_anterior}) ya no coincide con "
        f"'original_value' en config.yaml ({corr['original_value']}) — "
        "¿cambió el workbook fuente? Revisar antes de aplicar la corrección."
    )
    # Write the corrected value
    raw_events.loc[mask, corr["column"]] = corr["corrected_value"]
    # Log a warning so the correction is auditable
    logger.warning(
        "Corrección de dato aplicada: evento ID=%s, columna '%s': %s -> %s "
        "(decidido por %s, %s)",
        corr["event_id"], corr["column"], valor_anterior, corr["corrected_value"],
        corr["decided_by"], corr["decided_date"],
    )

# Log how many corrections were applied
logger.info("%d corrección(es) de datos aplicadas sobre raw_events", len(corrections))

## 2. Filtrado por alcance de peligro

Decisión 2026-08-05 (Jerson): el alcance de HCI v1 incluye el complejo de
peligros hidrometeorológicos relacionados con inundación — no solo
`EVENTO == "Inundación"` — para poder capturarlos también vía detección
remota en fases futuras (Sentinel-1, InSAR, etc.):

- `Inundación` (fluvial, pluvial/flash, costera)
- `Aluvión (Flujos)` (flujos de detritos asociados a lluvia intensa)
- `Deslizamiento`
- `Socavamiento`

El mapeo exacto vive en `config.yaml` → `hazard_scope`, no aquí, para poder
ajustarlo sin tocar código.

**English:** 2. Filter by hazard scope

In [ ]:
# hazard_scope mapea el valor textual de la columna EVENTO -> código interno
# ASCII normalizado, usado en el resto del pipeline (más robusto que
# comparar strings con tildes en cada módulo posterior).
# Mapping of raw event names to hazard types in scope
hazard_scope = config["hazard_scope"]  # dict, ver config.yaml

# Filtrar solo las filas cuyo EVENTO está dentro del alcance acordado.
# Keep only events inside the hazard scope
events = raw_events[raw_events["EVENTO"].isin(hazard_scope)].copy()  # .copy() evita SettingWithCopyWarning

# Crear la columna normalizada hazard_type a partir del mapeo.
# Add a standardized hazard type column
events["hazard_type"] = events["EVENTO"].map(hazard_scope)

# Registrar cuántas filas sobrevivieron el filtro, y el desglose por tipo.
# Log how many events are in scope
logger.info("Eventos dentro del alcance de peligro: %d de %d", len(events), len(raw_events))
# Show the number of events per hazard type
events["hazard_type"].value_counts()

## 3. Sub-clasificación del mecanismo de inundación/aluvión

La fuente no distingue explícitamente inundación fluvial (`riverine`) de
pluvial/flash (`pluvial_flash`) o costera (`coastal`). Se deriva un
heurístico a partir de palabras clave en la columna `CAUSA`
(`config.yaml` → `flood_subtype_keywords`).

**Limitación documentada:** es una heurística basada en texto libre, no un
campo estructurado — debe tratarse como aproximación, no como verdad de
terreno. Ver documento técnico, sección "Limitaciones".

**English:** 3. Flood mechanism sub-type

In [ ]:
# Extraer el diccionario de palabras clave desde la configuración.
# Keywords that identify each flood sub-type, from the config
subtype_keywords = config["flood_subtype_keywords"]  # dict: subtipo -> lista de keywords

# Classify the flooding mechanism from the free-text cause
def classify_flood_subtype(causa: str) -> str:
    """Clasifica el mecanismo de inundación/aluvión a partir del texto de CAUSA.

    Args:
        causa: valor de la columna CAUSA para una fila (puede ser NaN).

    Returns:
        Uno de: 'riverine', 'coastal', 'pluvial_flash', 'other', 'unknown'.

    English:
    Classify the flooding mechanism (for example river overflow or flash flood) from the free-text cause.

    Args:
        causa: Free-text cause reported for the event.

    Returns:
        The sub-type label, 'unknown' for non-text values, or 'other' when no keyword matches.
    """
    # Non-text causes cannot be classified
    if not isinstance(causa, str):  # CAUSA vacío/NaN -> no se puede clasificar
        # Return 'unknown' for missing causes
        return "unknown"
    # Lower-case the text to match keywords
    causa_lower = causa.lower()  # normalizar a minúsculas para el match de keywords
    # Check each sub-type in order
    for subtype, keywords in subtype_keywords.items():  # recorrer cada subtipo candidato
        # A sub-type matches if any of its keywords appears in the text
        if any(kw in causa_lower for kw in keywords):  # ¿alguna keyword aparece en el texto?
            # Return the first matching sub-type
            return subtype  # primer match gana (orden del YAML = prioridad)
    # Default label when no keyword matches
    return "other"  # CAUSA con texto pero sin match a ninguna keyword conocida

# Aplicar la clasificación solo a flood y debris_flow — landslide/scour no
# tienen "mecanismo de inundación" (no aplica el concepto).
# Flag flood-like events (floods and debris flows)
is_flood_like = events["hazard_type"].isin(["flood", "debris_flow"])  # máscara booleana
# Assign the sub-type only to flood-like events
events["flood_subtype"] = np.where(
    is_flood_like,  # condición
    events["CAUSA"].apply(classify_flood_subtype),  # valor si es flood/debris_flow
    None,  # valor si es landslide/scour (no aplica)
)

# Reportar el desglose de subtipos para revisión humana.
# Log that the sub-type distribution was computed
logger.info("Distribución de flood_subtype calculada")
# Show the sub-type distribution
events.loc[is_flood_like, "flood_subtype"].value_counts()

## 4. Fechas y ventana temporal del proyecto (2015–2025)

**English:** 4. Dates and project time window (2015-2025)

In [ ]:
# Convertir la columna de fecha a datetime real (algunas fuentes traen texto).
# Parse event dates, turning invalid dates into missing values
events["event_date"] = pd.to_datetime(events["FECHA DEL EVENTO"], errors="coerce")  # NaT si no parsea

# Extraer año y mes — se usarán para el join con el índice ENSO mensual.
# Extract the year
events["year"] = events["event_date"].dt.year  # entero, p. ej. 2015
# Extract the month
events["month"] = events["event_date"].dt.month  # entero 1-12

# Verificar que no haya fechas no parseables (NaT) — si las hay, se reportan
# pero NO se descartan silenciosamente (decisión: visibilidad > limpieza ciega).
# Count dates that could not be parsed
n_unparsed_dates = events["event_date"].isna().sum()  # conteo de fechas inválidas
# Log the number of unparseable dates
logger.info("Fechas no parseables: %d de %d filas", n_unparsed_dates, len(events))

# Restringir al rango declarado del proyecto (config.yaml -> date_range).
# First year of the analysis window
year_min = config["date_range"]["start_year"]  # 2015
# Last year of the analysis window
year_max = config["date_range"]["end_year"]  # 2025
# Flag events inside the analysis window
in_range = events["year"].between(year_min, year_max)  # máscara booleana
# Count events outside the window
n_out_of_range = (~in_range).sum()  # cuántas filas quedan fuera de 2015-2025
# Log the number of events outside the window
logger.info("Filas fuera del rango %d-%d: %d", year_min, year_max, n_out_of_range)

# Keep only events inside the window
events = events[in_range].copy()  # aplicar el recorte temporal
# Log the number of rows after the temporal filter
logger.info("Filas tras recorte temporal: %d", len(events))

## 5. Validación de coordenadas

Se valida que `LATITUD`/`LONGITUD` caigan dentro de un bounding box de
Ecuador continental + insular (Galápagos), definido en `config.yaml` →
`ecuador_bbox`. No se descartan las filas fuera de rango — se marcan con
`coord_valid = False` para que fases posteriores decidan cómo tratarlas
(p. ej. usar centroide cantonal en su lugar).

**English:** 5. Coordinate validation

In [ ]:
# Leer los límites del bounding box desde la configuración.
# Bounding box of Ecuador from the config
bbox = config["ecuador_bbox"]  # dict con lat_min, lat_max, lon_min, lon_max

# Evaluar si cada coordenada cae dentro del bbox esperado.
# Check latitude is inside the bounding box
lat_ok = events["LATITUD"].between(bbox["lat_min"], bbox["lat_max"])  # True/False por fila
# Check longitude is inside the bounding box
lon_ok = events["LONGITUD"].between(bbox["lon_min"], bbox["lon_max"])  # True/False por fila
# A coordinate is valid when both latitude and longitude are inside
events["coord_valid"] = lat_ok & lon_ok  # válida solo si ambas condiciones se cumplen

# Reportar cuántas coordenadas son nulas o inválidas — insumo directo para
# la sección de limitaciones del documento técnico.
# Count missing coordinates
n_null_coords = events["LATITUD"].isna().sum() + events["LONGITUD"].isna().sum()
# Count coordinates outside the bounding box
n_invalid_coords = (~events["coord_valid"]).sum()
# Log both counts
logger.info("Coordenadas nulas: %d | Coordenadas fuera de bbox Ecuador: %d", n_null_coords, n_invalid_coords)

# Show how many events have valid coordinates
events["coord_valid"].value_counts()

## 6. Tratamiento de información de identificación personal (PII)

La columna `Fuente` contiene nombres y números telefónicos de operadores
SNGRE/ECU911 (verificado: ~81% de las filas del alcance filtrado). Por
gobernanza de datos de un producto institucional FAO, **no se conserva el
texto crudo** en la tabla procesada — se reemplaza por un indicador booleano
y un tipo de fuente genérico. El texto original permanece intacto en el
workbook crudo (`data/raw`), nunca se modifica la fuente.

**English:** 6. Personal data (PII) handling

In [ ]:
# Expresión regular para detectar patrones de teléfono ecuatoriano/genérico
# dentro del texto libre de la columna Fuente (config.yaml -> pii.phone_regex).
phone_pattern = config["pii"]["phone_regex"]  # p. ej. '09\\d{8}|\\d{7,}'

# Marcar qué filas parecen contener un número telefónico en Fuente.
# Flag events whose source text may contain personal data
events["source_has_pii"] = events["Fuente"].astype(str).str.contains(
    phone_pattern, regex=True, na=False
)  # True/False por fila

# Reportar el porcentaje de filas con PII detectada, antes de eliminarla.
# Percentage of events flagged
pct_pii = events["source_has_pii"].mean() * 100  # porcentaje
# Log the percentage
logger.info("Filas con posible PII en 'Fuente': %.1f%%", pct_pii)

# Generalizar la fuente a un tipo institucional fijo, sin datos personales.
# (Todas las filas de BD_2015_2025 provienen de reportes SNGRE/ECU911 — ver
# 'Notas Metodológicas' del workbook maestro.)
# Record the source type of every event
events["source_type"] = "SNGRE_ECU911_report"  # constante — reemplaza el texto crudo

# Eliminar la columna cruda con PII de la tabla procesada, según config.yaml
# -> pii.drop_raw_source_text.
# Drop the free-text source column if the config says so
if config["pii"]["drop_raw_source_text"]:
    # Remove the column that may contain personal data
    events = events.drop(columns=["Fuente"])  # el texto original sigue en data/raw, intacto
    # Log that the column was removed
    logger.info("Columna 'Fuente' (texto libre con PII) eliminada de la tabla procesada")

## 7. Verificación de duplicados

Se valida unicidad del identificador `ID` dentro del subconjunto filtrado —
un duplicado indicaría un problema de exportación de la fuente, no algo a
corregir silenciosamente.

**English:** 7. Duplicate check

In [ ]:
# Contar cuántos IDs están duplicados dentro de la tabla ya filtrada.
# Count duplicated event IDs
n_duplicated_ids = events["ID"].duplicated().sum()  # 0 esperado
# Log the number of duplicates
logger.info("IDs duplicados dentro del alcance filtrado: %d", n_duplicated_ids)

# Si hubiera duplicados, se detienen aquí para revisión manual — no se
# eliminan automáticamente sin que quede trazado en el log.
# Warn when duplicates exist
if n_duplicated_ids > 0:
    # Warning asking for review before continuing
    logger.warning("Se detectaron IDs duplicados — revisar antes de continuar")

## 8. Construcción del índice ENSO mensual completo (`ENSO_Index_Monthly`)

Se usa la hoja `2015 2025 (mes)` del workbook maestro **solo como referencia
puntual** — se comprobó que NO cubre los 132 meses de 2015-2025, sino
únicamente los 38-52 meses con impacto ENSO documentado por el equipo COE
(ver `Notas Metodológicas`). Para tener una serie mensual **completa**, se
reconstruye desde las hojas `ICEN_Trimestral` y `ONI trimestral` (formato
largo, 1950–presente), usando la convención de trimestre móvil validada
empíricamente contra la hoja curada (ver documento técnico, sección
"Validación de la convención trimestre→mes").

**English:** 8. Complete monthly ENSO series

In [ ]:
# Mapeo trimestre ENSO (3 meses móviles) -> mes calendario central,
# validado empíricamente (ver documento técnico). Vive en config.yaml.
trimester_to_month = config["trimester_to_month"]  # p. ej. {'DEF': 1, 'EFM': 2, ...}

# Leer las series trimestrales largas de ICEN y ONI.
# Read the quarterly ICEN sheet
icen_trimestral = read_excel_sheet(workbook_path, config["sheets"]["icen_trimestral"])
# Read the quarterly ONI sheet
oni_trimestral = read_excel_sheet(workbook_path, config["sheets"]["oni_trimestral"])

# Eliminar filas sin año (filas de encabezado/vacías arrastradas del Excel).
# Drop ICEN rows without a year
icen_trimestral = icen_trimestral.dropna(subset=["Año"])  # limpieza mínima necesaria
# Drop ONI rows without a year
oni_trimestral = oni_trimestral.dropna(subset=["Año"])

# Traducir el código de trimestre (texto) al mes calendario central (entero).
# Convert ICEN quarters to months
icen_trimestral["month"] = icen_trimestral["Trimestre"].map(trimester_to_month)
# Convert ONI quarters to months
oni_trimestral["month"] = oni_trimestral["Trimestre"].map(trimester_to_month)

# Asegurar tipo entero en el año (a veces llega como float por celdas vacías).
# Integer year for ICEN
icen_trimestral["year"] = icen_trimestral["Año"].astype(int)
# Integer year for ONI
oni_trimestral["year"] = oni_trimestral["Año"].astype(int)

In [ ]:
# Unir ICEN y ONI en una sola tabla mensual, por año+mes (outer join: no se
# pierde ningún mes aunque falte en una de las dos fuentes).
# Start the monthly series from ICEN with standardized column names
enso_monthly = icen_trimestral[["year", "month", "ICEN", "Categoría y magnitud"]].rename(
    columns={"Categoría y magnitud": "icen_category"}  # nombre explícito, sin ambigüedad
).merge(
    oni_trimestral[["year", "month", "ONI", "Categoría y magnitud"]].rename(
        columns={"Categoría y magnitud": "oni_category"}
    ),
    on=["year", "month"],  # llave de unión
    how="outer",  # conservar todos los meses de ambas fuentes
)

# Recortar a la ventana temporal del proyecto (2015-2025).
# Keep only the months inside the analysis window
enso_monthly = enso_monthly[
    (enso_monthly["year"] >= year_min) & (enso_monthly["year"] <= year_max)
].sort_values(["year", "month"]).reset_index(drop=True)

# Validación de completitud: se esperan 11 años x 12 meses = 132 filas.
# Expected number of months: 12 per year
expected_rows = (year_max - year_min + 1) * 12  # 132
# Log the length of the monthly series
logger.info(
    "ENSO_Index_Monthly: %d filas (esperado %d) | nulos ICEN=%d ONI=%d",
    len(enso_monthly), expected_rows,
    enso_monthly["ICEN"].isna().sum(), enso_monthly["ONI"].isna().sum(),
)
# Check the monthly series is complete
assert len(enso_monthly) == expected_rows, "Serie ENSO mensual incompleta — revisar fuente"

# Preview the first 12 months
enso_monthly.head(12)

## 9. Enriquecimiento de eventos con el índice ENSO mensual

Join por `year` + `month`: cada evento queda vinculado al valor y categoría
ICEN/ONI del mes en que ocurrió. Este notebook **no** calcula todavía el
criterio de cohorte (OR/AND) ni el Relative Risk — eso corresponde a la
Fase 3 (indicadores), para no mezclar responsabilidades entre fases.

**English:** 9. Join events to the monthly ENSO series

In [ ]:
# Left join: conservar TODOS los eventos, aunque (no debería pasar, ya que
# la serie ENSO ahora es completa 2015-2025) alguno quedara sin match.
# Join each event to the ENSO state of its month
events = events.merge(enso_monthly, on=["year", "month"], how="left")

# Tasa de emparejamiento — debe ser 1.0 (100%) dado que la serie ENSO ya
# cubre los 132 meses completos del rango del proyecto.
# Share of events that received an ENSO value
match_rate = events["ICEN"].notna().mean()
# Log the match rate
logger.info("Tasa de emparejamiento evento-ENSO: %.1f%%", match_rate * 100)
# Check every event matched an ENSO month
assert match_rate == 1.0, "Hay eventos sin ENSO asociado — revisar rango de fechas"

## 10. Selección y renombrado final de columnas (esquema `Hazard_Event`)

Se seleccionan las columnas relevantes para el esquema `Flood_Event`
ampliado a `Hazard_Event` (ver diseño Fase 0, sección 7, y su extensión del
2026-08-05 para el complejo de peligros hidrometeorológicos).

**English:** 10. Final column selection and renaming

In [ ]:
# Diccionario de renombrado: nombre crudo de SNGRE -> nombre estandarizado
# del esquema Hazard_Event (snake_case, sin tildes, en inglés para el core
# schema — facilita reutilización en otros países, ver Fase 0 §17).
# Mapping from raw column names to the final schema
column_mapping = {
    "ID": "event_id",
    "PROVINCIA": "province",
    "CODIFICACION PROVINCIAL": "province_code",
    "CANTON": "canton",
    "CODIFICACION CANTONAL": "canton_code",
    "PARROQUIAS": "parish",
    "CODIFICACION PARROQUIAL": "parish_code",
    "LATITUD": "latitude",
    "LONGITUD": "longitude",
    "CAUSA": "cause_raw",
    "FALLECIDAS": "deaths",
    "HERIDAS": "injured",
    "PERSONAS DESAPARECIDAS": "missing_persons",
    "PERSONAS AFECTADAS DIRECTAMENTE": "affected_direct",
    "AFECTADAS INDIRECTAS": "affected_indirect",
    "PERSONAS DAMNIFICADAS": "affected_severe",
    "PERSONAS EVACUADAS": "evacuated",
    "PERSONAS ALBERGADAS": "sheltered",
    "VIVIENDAS AFECTADAS": "houses_affected",
    "VIVIENDAS DESTRUIDAS": "houses_destroyed",
    "ESTABLECIMIENTOS EDUCATIVOS AFECTADOS": "schools_affected",
    "ESTABLECIMIENTOS EDUCATIVOS DESTRUIDOS": "schools_destroyed",
    "CENTROS DE SALUD AFECTADOS": "health_facilities_affected",
    "CENTROS DE SALUD DESTRUIDOS": "health_facilities_destroyed",
    "Ha. CULTIVO AFECTADAS": "crops_affected_ha",
    "Ha. CULTIVO PERDIDAS": "crops_destroyed_ha",
    "ANIMALES AFECTADOS": "livestock_affected",
    "ANIMALES MUERTOS": "livestock_deaths",
}

# Columnas ya creadas en pasos anteriores que se conservan tal cual.
# Columns created in this notebook
own_columns = [
    "event_date", "year", "month", "hazard_type", "flood_subtype",
    "coord_valid", "source_has_pii", "source_type",
    "ICEN", "icen_category", "ONI", "oni_category",
]

# Construir la lista final de columnas crudas a conservar (las que sí están
# en column_mapping), en el orden del diccionario.
# Raw columns to keep
raw_cols_to_keep = list(column_mapping.keys())  # nombres originales de SNGRE

# Sub-seleccionar y renombrar en un solo paso.
# Select and rename columns to the final schema
hazard_event = events[raw_cols_to_keep + own_columns].rename(columns=column_mapping)

# Log the size of the final table
logger.info("Tabla Hazard_Event final: %d filas x %d columnas", *hazard_event.shape)
# Preview the first three rows
hazard_event.head(3)

## 11. Resumen de calidad de datos (para el documento técnico y el reporte de fase)

**English:** 11. Data-quality summary

In [ ]:
# Resumen consolidado — estos números alimentan directamente la sección de
# "Validación de resultados" del documento técnico y el resumen de fase.
# Dictionary with the data-quality summary
summary = {
    "total_eventos_alcance": len(hazard_event),
    "rango_fechas": (str(hazard_event["event_date"].min()), str(hazard_event["event_date"].max())),
    "por_hazard_type": hazard_event["hazard_type"].value_counts().to_dict(),
    "coordenadas_invalidas_pct": round((~hazard_event["coord_valid"]).mean() * 100, 2),
    "provincias_unicas": hazard_event["province"].nunique(),
    "cantones_unicos": hazard_event["canton_code"].nunique(),
    "tasa_emparejamiento_enso_pct": 100.0,  # validado por el assert de la sección 9
    "filas_con_pii_removida_pct": round(hazard_event["source_has_pii"].mean() * 100, 2),
}

# Print each summary item
for key, value in summary.items():  # imprimir cada métrica de forma legible
    # Print the key and its value
    print(f"{key}: {value}")

# Log the quality summary
logger.info("Resumen de calidad de datos generado: %s", summary)

## 12. Exportación

Se exportan dos tablas a `data/processed/`, en Parquet (consumo por el resto
del pipeline) y CSV (inspección manual):

- `hazard_event.parquet` / `.csv` — eventos limpios y armonizados.
- `enso_index_monthly.parquet` / `.csv` — serie ENSO mensual completa 2015-2025.

**English:** 12. Export

In [ ]:
# Directorio de salida resuelto desde config.yaml (relativo a notebooks/).
# Folder for processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())

# Exportar la tabla de eventos armonizada.
# Write the hazard events table
hazard_event_paths = write_processed_table(hazard_event, processed_dir, "hazard_event")

# Exportar también la serie ENSO mensual completa, reutilizable por
# cualquier fase futura sin tener que releer el workbook de Excel.
# Write the monthly ENSO series
enso_monthly_paths = write_processed_table(enso_monthly, processed_dir, "enso_index_monthly")

# Log the exported files
logger.info("Exportación completa: %s", {**hazard_event_paths, **enso_monthly_paths})
# Print the export folder
print("Archivos escritos en:", processed_dir)

## 13. Próximos pasos (Fase 2)

1. Integración espacial: unir `hazard_event` con límites administrativos
   oficiales (INEC/OCHA) vía `canton_code`/Pcode, exportar GeoPackage.
2. Resolver la fuente exacta de límites administrativos (INEC vs. OCHA) —
   decisión abierta desde el diseño Fase 0 (§19.4).
3. Contrastar el conteo de eventos de esta tabla contra el modelo existente
   de "Escenarios y Nivel de Respuesta" del workbook maestro, como
   validación externa (acordado 2026-08-05).
4. Documentar el criterio exacto de filtro de `Base2` (aún sin resolver) por
   si aporta señal de calidad adicional para descartar/marcar eventos.
